# Chapter 15.3 멀티에이전트 RL — 멀티에이전트 학습 동역학

[![Open In Colab: 멀티에이전트 학습 동역학](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter15_3_multiagent_dynamics.ipynb)

책 본문: [15.3 멀티에이전트 RL 개관](https://smhanlab.com/book-ml/kor/ml2/chapter15/3.html)

이 노트북은 책 15.3절의 세 실습(죄수의 딜레마, 조율 게임, 바위-가위-보)을
그대로 실행합니다. 핵심 관점: **학습 알고리즘은 세 실습 모두
ε-greedy Q-learning(또는 fictitious play)으로 똑같고, 결과가 완전히
달라지는 것은 오직 보상 구조(게임의 규칙)**입니다.


## 1. 설정: ε-greedy Q-learning self-play

각 에이전트는 Chapter 6의 Q-learning을 그대로 수행할 뿐이다 —
"상대도 학습 중"이라는 사실은 알고리즘 어디에도 명시되지 않는다.
`opp_fixed`를 주면 상대는 학습하지 않는 고정 에이전트가 된다(대조 실험).


In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"
import numpy as np

# 행동: 0 = C(협력/침묵), 1 = D(배신/자백).  행=에이전트1, 열=에이전트2
PD = np.array([[3.0, 0.0],   # (C,C)=3, (C,D)=0
               [5.0, 1.0]])  # (D,C)=5, (D,D)=1

def eps_greedy(Q, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(len(Q)))
    return int(np.argmax(Q))

def self_play(games, payoff, alpha=0.5, eps_start=1.0, eps_end=0.02,
              tau=500.0, seed=0, opp_fixed=None):
    """둘 다 ε-greedy Q-learning으로 배우며 서로 플레이(self-play).
    opp_fixed를 주면 상대는 그 행동을 고정(학습 X, 대조 실험용)."""
    rng = np.random.default_rng(seed)
    Q1, Q2 = np.zeros(2), np.zeros(2)
    rewards1, outcomes = [], []
    for g in range(games):
        eps = eps_end + (eps_start - eps_end) * np.exp(-g / tau)  # 탐험 감소
        a1 = eps_greedy(Q1, eps, rng)
        a2 = opp_fixed if opp_fixed is not None else eps_greedy(Q2, eps, rng)
        r1 = payoff[a1, a2]
        r2 = payoff[a2, a1]               # 게임이 대칭
        Q1[a1] += alpha * (r1 - Q1[a1])   # Chapter 6의 TD 갱신 그대로
        if opp_fixed is None:
            Q2[a2] += alpha * (r2 - Q2[a2])
        rewards1.append(r1)
        outcomes.append((a1, a2))
    return np.array(rewards1), np.array(outcomes), Q1

## 2. 실습 1: 죄수의 딜레마 — 둘 다 "정확히" 배우면?

지배전략이 D이므로 각자의 "합리적인" 학습이 쌓이면 (D,D) —
**공동으로 열위한 내쉬 균형** — 으로 수렴한다. 알고리즘이 아니라
게임이 결과를 정한다.


In [2]:
GAMES = 20000
rew, out, Q1 = self_play(GAMES, PD, seed=0)

print(f"초기 500경기 평균 보상      = {rew[:500].mean():.2f}  (C·D 섞인 탐색기)")
print(f"마지막 1,000경기 평균 보상  = {rew[-1000:].mean():.3f}  (이론값 1.0)")
print(f"2,500경기 이후 (D,D) 비율   = {np.mean(out[2500:] == (1, 1)):.1%}")
print(f"학습된 Q1 = {np.round(Q1, 3)}  → argmax = {'C' if Q1[0] > Q1[1] else 'D'}")
print()
print("→ '완벽하게' Q-learning한 둘이 공동으로 (C,C)보다 나쁜 (D,D)에 수렴")


초기 500경기 평균 보상      = 1.96  (C·D 섞인 탐색기)
마지막 1,000경기 평균 보상  = 1.012  (이론값 1.0)
2,500경기 이후 (D,D) 비율   = 99.0%
학습된 Q1 = [0. 1.]  → argmax = D

→ '완벽하게' Q-learning한 둘이 공동으로 (C,C)보다 나쁜 (D,D)에 수렴


In [3]:
# 대조 실험: 상대를 '항상 C(협력)' 고정 에이전트로
rew_c, out_c, Q1_c = self_play(GAMES, PD, seed=0, opp_fixed=0)
print(f"[대조] 상대가 항상 C: 마지막 1,000경기 평균 보상 = {rew_c[-1000:].mean():.3f}  (이론값 5.0)")
print(f"학습된 Q1 = {np.round(Q1_c, 3)} → 협력하는 상대를 착취해 D(배신)로 수렴")
print()
print("→ (D,D)로 가든 상대를 착취하는 D로 가든, 결정하는 것은 알고리즘이 아니라 게임")


[대조] 상대가 항상 C: 마지막 1,000경기 평균 보상 = 4.976  (이론값 5.0)
학습된 Q1 = [3. 5.] → 협력하는 상대를 착취해 D(배신)로 수렴

→ (D,D)로 가든 상대를 착취하는 D로 가든, 결정하는 것은 알고리즘이 아니라 게임


## 3. 실습 2: 조율 게임 — 두 내쉬 균형, 어느 쪽에 "잠기느냐"

(L,L)도 (R,R)도 내쉬 균형이다. 알고리즘은 둘 다 찾아내지만
**어느 쪽으로 수렴하느냐는 초기 탐색의 잡음(시드)이 결정**한다 —
경로 의존(path dependence).


In [4]:
COORD = np.array([[2.0, 0.0],   # (L,L)=2, (L,R)=0
                  [0.0, 2.0]])  # (R,L)=0, (R,R)=2
for seed in range(8):
    rew, out, _ = self_play(GAMES, COORD, seed=seed)
    same = out[-1000:, 0] == out[-1000:, 1]
    if same.mean() > 0.9:
        which = 'L' if (out[-1000:, 0] == 0).mean() > 0.5 else 'R'
        print(f"seed={seed}: 같은 방향 {same.mean():.1%} → ({which},{which}) 균형에 잠금(lock-in)")
    else:
        print(f"seed={seed}: 같은 방향 {same.mean():.1%} → 아직 잠김 안 됨")
print()
print("→ 같은 알고리즘, 시드만 달라서 다른 균형에 잠긴다")


seed=0: 같은 방향 98.4% → (L,L) 균형에 잠금(lock-in)
seed=1: 같은 방향 98.6% → (L,L) 균형에 잠금(lock-in)


seed=2: 같은 방향 98.0% → (R,R) 균형에 잠금(lock-in)


seed=3: 같은 방향 97.8% → (L,L) 균형에 잠금(lock-in)


seed=4: 같은 방향 97.7% → (R,R) 균형에 잠금(lock-in)
seed=5: 같은 방향 97.4% → (L,L) 균형에 잠금(lock-in)


seed=6: 같은 방향 98.3% → (R,R) 균형에 잠금(lock-in)


seed=7: 같은 방향 97.9% → (R,R) 균형에 잠금(lock-in)

→ 같은 알고리즘, 시드만 달라서 다른 균형에 잠긴다


## 4. 실습 3: 바위-가위-보 — 순수 전략 균형이 없는 게임

Q-learning은 순수 전략에 잠기려는 성향이 있어 이 균형을 재현하지
못한다. 제로섬 게임에서는 **fictitious play**가 쓰인다 — 상대의
행동 경험 빈도에 best response만 반복하면, 그 반복의 한계가 바로
혼합 내쉬 균형 (1/3, 1/3, 1/3)이다.


In [5]:
# 바위(0)>가위(1), 가위(1)>보(2), 보(2)>바위(0)
RPS = np.array([[0, 1, -1],
                [-1, 0, 1],
                [1, -1, 0]], dtype=float)

def fictitious_play(games, payoff, seed=0, eps=0.1, history=False):
    """제로섬: 각 에이전트는 상대 행동의 경험 빈도에 best response.
    eps 확률로 균등 랜덤을 섞어 순수 BR의 고착을 피한다."""
    rng = np.random.default_rng(seed)
    n = payoff.shape[0]
    freq1 = np.ones(n) / n   # 에이전트1 행동의 누적 빈도
    freq2 = np.ones(n) / n   # 에이전트2 행동의 누적 빈도
    hist1, hist2 = [], []
    for t in range(games):
        br1 = int(np.argmax(payoff @ freq2))     # 1: 상대 빈도에 BR
        br2 = int(np.argmin(payoff.T @ freq1))   # 2: zero-sum → 1의 기대보상 최소화
        a1 = br1 if rng.random() < (1 - eps) else int(rng.integers(n))
        a2 = br2 if rng.random() < (1 - eps) else int(rng.integers(n))
        freq1 = (t * freq1 + np.eye(n)[a1]) / (t + 1)
        freq2 = (t * freq2 + np.eye(n)[a2]) / (t + 1)
        if history:
            hist1.append(freq1.copy()); hist2.append(freq2.copy())
    if history:
        return freq1, freq2, np.array(hist1), np.array(hist2)
    return freq1, freq2

f1, f2 = fictitious_play(2000, RPS, seed=0)
print(f"최종 행동 빈도: 에이전트 1 = {np.round(f1, 3)}")
print(f"               에이전트 2 = {np.round(f2, 3)}")
print("→ 둘 다 혼합 내쉬 균형 (1/3, 1/3, 1/3)으로 수렴")


최종 행동 빈도: 에이전트 1 = [0.326 0.331 0.344]
               에이전트 2 = [0.342 0.338 0.32 ]
→ 둘 다 혼합 내쉬 균형 (1/3, 1/3, 1/3)으로 수렴


In [6]:
# 확인: eps = 0(순수 best response만)이면? (확인 문제 3번의 소재)
f1_0, f2_0 = fictitious_play(2000, RPS, seed=0, eps=0.0)
print(f"[eps=0] 에이전트 1 = {np.round(f1_0, 3)}   에이전트 2 = {np.round(f2_0, 3)}")
print("시드·조건에 따라 순수 BR만으론 혼합 균형에 안착하지 않을 수 있다 —")
print("작은 랜덤성(eps=0.1)의 역할을 확인 문제 3번에서 생각해보자.")


[eps=0] 에이전트 1 = [0.332 0.322 0.346]   에이전트 2 = [0.332 0.322 0.346]
시드·조건에 따라 순수 BR만으론 혼합 균형에 안착하지 않을 수 있다 —
작은 랜덤성(eps=0.1)의 역할을 확인 문제 3번에서 생각해보자.


## 5. 그림: 세 학습 동역학 + 대조 실험 (책 15.3절의 4패널 그림)

동일한 학습 규칙이 **보상 행렬에 따라** 완전히 다른 동역학을
만든다 — (a) 상호 배신, (b) 착취, (c) 혼합 균형, (d) 균형 잠금.


In [7]:
def moving_avg(x, w=100):
    return np.convolve(x, np.ones(w) / w, mode='valid')

games = np.arange(GAMES)
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# (a) 죄수의 딜레마 self-play
ax = axes[0, 0]
ax.plot(games[99:], moving_avg(rew), lw=1)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.axhline(3.0, color='gray', ls=':', lw=1)
ax.set_title(f"(a) 죄수의 딜레마 self-play → (D,D) [최종 평균 {rew[-1000:].mean():.3f}]")
ax.set_xlabel('경기'); ax.set_ylabel('평균 보상 (이동평균)')
ax.set_ylim(0, 4)

# (b) 상대가 항상 C인 고정 에이전트
ax = axes[0, 1]
ax.plot(games[99:], moving_avg(rew_c), lw=1)
ax.axhline(5.0, color='gray', ls='--', lw=1)
ax.set_title(f"(b) 상대 항상 C → 학습자는 D로 수렴 [{rew_c[-1000:].mean():.3f}]")
ax.set_ylim(0, 6)

# (c) 바위-가위-보 fictitious play
ax = axes[1, 0]
_, _, H1, H2 = fictitious_play(2000, RPS, seed=0, history=True)
t = np.arange(1, 2001)
ax.plot(t, H1.mean(axis=1), lw=1, label='에이전트 1 (행동 빈도 평균)')
ax.plot(t, H2.mean(axis=1), lw=1, label='에이전트 2 (행동 빈도 평균)')
ax.axhline(1/3, color='gray', ls='--', lw=1)
ax.set_title("(c) 바위-가위-보 fictitious play → 혼합 균형 1/3")
ax.set_xlabel('경기'); ax.set_ylabel('행동 빈도 (3행동의 평균)')

# (d) 조율 게임 self-play
ax = axes[1, 1]
same_frac = (out[:, 0] == out[:, 1]).astype(float)
ax.plot(games[99:], moving_avg(same_frac), lw=1)
ax.set_title(f"(d) 조율 게임 self-play → 하나의 균형에 잠금 [같은 방향 {same_frac[-1000:].mean():.1%}]")
ax.set_xlabel('경기'); ax.set_ylabel('같은 방향 선택 비율 (이동평균)')
ax.set_ylim(0, 1)

fig.suptitle("멀티에이전트 학습의 세 가지 동역학 — 알고리즘은 같고, 보상 행렬만 달랐다", y=1.02)
fig.tight_layout()
fig.savefig(IMG + "/ch15_3_marl_dynamics.svg", bbox_inches="tight")
plt.show()
print(f"그림 저장: {IMG}/ch15_3_marl_dynamics.svg")

그림 저장: /home/smhan/book-ml/kor/src/images/ch15_3_marl_dynamics.svg
